# FIUBench PO-forget5 — Kaggle T4 ×2
위에서 아래로 실행. Internet On, T4 ×2. Batch 4/GPU, accumulation 16, effective forget batch 128 (PO retain 포함 256 sequences), 8 epochs.
핵심 버전은 이전 smoke test에서 확인한 조합. 보조 패키지는 전체 lockfile이 없으므로 설치 후 freeze를 보관합니다. 이 노트북은 학습용이며 evaluation 검증은 포함하지 않습니다.
Python 원본 변경은 Kaggle 복사본에만 적용합니다. 저장은 8 optimizer steps마다 및 종료 시. checkpoint.pt는 학습 파라미터만 저장하며 optimizer 재개 상태는 없습니다.


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d FIUBench_kaggle ]; then
  git clone https://github.com/npngjjm/FIUBench_kaggle.git
fi
cd FIUBench_kaggle
git rev-parse HEAD
nvidia-smi
apt-get update -qq
apt-get install -y libaio-dev
python -m pip uninstall -y torchao
python -m pip install 'transformers==4.37.2' 'tokenizers==0.15.1' 'peft==0.8.2' 'accelerate==1.14.0' 'deepspeed==0.19.7' 'huggingface_hub==0.36.2' 'numpy==2.1.3' hydra-core omegaconf datasets sentencepiece protobuf scipy ninja pillow pyyaml requests tqdm

설치 후 Python 커널을 재시작하고 다음 셀부터 실행하세요. Kaggle 기본 PyTorch는 교체하지 않습니다. 기존 패키지 의존성 경고를 없애기 위해 무작정 버전을 올리지 마세요.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from importlib.metadata import version
import torch, transformers, tokenizers, peft, accelerate, deepspeed
repo = Path('/kaggle/working/FIUBench_kaggle')
os.chdir(repo)
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
expected = {'transformers':'4.37.2', 'tokenizers':'0.15.1', 'peft':'0.8.2', 'accelerate':'1.14.0', 'deepspeed':'0.19.7', 'huggingface-hub':'0.36.2', 'numpy':'2.1.3'}
for name, wanted in expected.items():
    print(name, version(name))
    assert version(name) == wanted
print(sys.version, torch.__version__, torch.version.cuda)
assert torch.__version__ == '2.11.0+cu128', '검증한 PyTorch와 다릅니다. 환경을 확인하세요.'
assert torch.cuda.device_count() == 2
assert all('T4' in torch.cuda.get_device_name(i) for i in range(2))
print('핵심 환경 확인 완료')

In [ ]:
# 저장소 JSON과 prompt는 유지하고 SFHQ.zip만 공식 데이터 저장소에서 받습니다.
import json, shutil, zipfile
from collections import defaultdict
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
repo = Path('/kaggle/working/FIUBench_kaggle')
for name in ['full.json','split.json','prompt.json']:
    assert (repo/'dataset'/name).is_file(), name
records = [json.loads(s) for s in (repo/'dataset/full.json').read_text().splitlines() if s.strip()]
revision = HfApi().dataset_info('gray311/FIUBench').sha
archive = hf_hub_download('gray311/FIUBench', 'SFHQ.zip', repo_type='dataset', revision=revision, local_dir='/kaggle/working/fiu_download')
# ZIP 내부 상위 폴더명에 의존하지 않고, JSON이 요구하는 파일명을 대조합니다.
with zipfile.ZipFile(archive) as z:
    entries = defaultdict(list)
    for member in z.infolist():
        p = Path(member.filename)
        if not member.is_dir() and '__MACOSX' not in p.parts and not p.name.startswith('._'):
            entries[p.name].append(member)
    for record in records[:400]:
        target = (repo/record['image_path']).resolve()
        assert target.is_relative_to((repo/'dataset').resolve())
        matches = entries[target.name]
        assert len(matches) == 1, (target.name, len(matches))
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(matches[0]) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)
        with Image.open(target) as im:
            im.verify()
missing = [r['image_path'] for r in records[:400] if not (repo/r['image_path']).is_file()]
assert len(records[:400]) == 400 and not missing
Path('/kaggle/working/fiu_data_revision.txt').write_text(revision)
print('학습 대상 400개 레코드의 이미지 경로 및 이미지 파일 검사 통과', revision)

In [ ]:
from pathlib import Path
repo = Path('/kaggle/working/FIUBench_kaggle')
# 로컬 수정이 아직 GitHub에 push되지 않았어도 같은 config를 사용합니다.
(repo/'config/forget_po_forget5.yaml').write_text('# Table 7: batch 8, accumulation 16; released launcher uses one process.\n# T4 x2: 4 per GPU * 2 GPUs * 16 accumulation = 128 forget examples.\ndefaults:\n  - forget_lora\n  - _self_\n\nmodel_family: llava-phi\nmodel_path: gray311/vlm_unlearning_ft_llava_phi_3_mini\nattn_implementation: sdpa\ndata_path: ./dataset/full.json\nsplit: forget5\nforget_loss: idk\nlr: 3e-4\nbatch_size: 4\ngradient_accumulation_steps: 16\nnum_epochs: 8\nLoRA:\n  r: 128\n  alpha: 256\n  dropout: 0.05\nworkers: 0\nreport_to: null\nsave_dir: /kaggle/working/models/llava_phi_PO_forget5\nsave_model: true\nsave_steps: 8\n')
import yaml
a = yaml.safe_load((repo/'config/accelerate_kaggle_t4x2.yaml').read_text())
assert a['num_processes'] == 2
assert a['deepspeed_config']['gradient_accumulation_steps'] == 16
assert a['mixed_precision'] == 'fp16'
print((repo/'config/forget_po_forget5.yaml').read_text())

In [ ]:
# Kaggle 복사본에만 적용하는 호환성 패치. 로컬/GitHub .py는 수정하지 않습니다.
import ast
from pathlib import Path
path = Path('/kaggle/working/FIUBench_kaggle/forget.py')
text = path.read_text()
original = text
old = 'AutoTokenizer.from_pretrained(cfg.model_path)'
new = 'AutoTokenizer.from_pretrained(cfg.model_path, use_fast=False)'
assert old in text or new in text
text = text.replace(old, new)
marker = '    if cfg.LoRA.r != 0:\n        config = LoraConfig('
block = '''    if str(model_cfg.get("gradient_checkpointing", "false")).lower() == "true":
        model.config.use_cache = False
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
        logger.info("Gradient checkpointing enabled", main_process_only=True)

'''
if 'model.gradient_checkpointing_enable()' not in text:
    assert text.count(marker) == 1
    text = text.replace(marker, block + marker, 1)
end = text.rfind('\nif __name__ == "__main__":')
assert end > 0
if 'if cfg.save_model and accelerator.is_main_process:' not in text:
    start = text.rfind('\n    accelerator.end_training()', 0, end)
    assert start > 0
    text = text[:start] + '''
    output_dir = cfg.save_dir
    accelerator.wait_for_everyone()
    if cfg.save_model and accelerator.is_main_process:
        os.makedirs(output_dir, exist_ok=True)
        unwrapped_model = accelerator.unwrap_model(model)
        if cfg.LoRA.r != 0:
            save_lora_weights(unwrapped_model, output_dir)
        else:
            unwrapped_model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
    accelerator.wait_for_everyone()
    accelerator.end_training()
''' + text[end:]
ast.parse(text)
backup = path.with_suffix('.py.before_kaggle_patch')
if not backup.exists():
    backup.write_text(original)
path.write_text(text)
assert 'use_fast=False' in text and 'model.enable_input_require_grads()' in text
print('런타임 패치 및 Python 구문 검사 완료')

In [ ]:
import os, subprocess
from pathlib import Path
from hydra import compose, initialize_config_dir
from data_module import MMForgetDatasetQA, custom_data_collator_forget
repo = Path('/kaggle/working/FIUBench_kaggle')
os.chdir(repo)
with initialize_config_dir(version_base=None, config_dir=str(repo/'config')):
    cfg = compose(config_name='forget_po_forget5')
assert cfg.batch_size * 2 * cfg.gradient_accumulation_steps == 128
assert cfg.forget_loss == 'idk' and cfg.split == 'forget5'
assert cfg.num_epochs == 8 and cfg.save_steps == 8
assert cfg.attn_implementation == 'sdpa'
assert not (Path(cfg.save_dir)/'checkpoint.pt').exists(), '기존 최종 결과가 있습니다. 새 save_dir을 지정하세요.'
Path('/kaggle/working/fiu_environment.txt').write_bytes(subprocess.check_output(['python','-m','pip','freeze']))
Path('/kaggle/working/fiu_runtime.patch').write_bytes(subprocess.check_output(['git','diff','--','forget.py']))
print('Effective forget batch = 128; PO retain 포함 = 256 sequences/full update')
print('설정상 총 optimizer steps = 32; 저장 주기 = 8')

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/FIUBench_kaggle
export HF_HOME=/kaggle/working/hf_cache
export TOKENIZERS_PARALLELISM=false
export HYDRA_FULL_ERROR=1
export PYTHONUNBUFFERED=1
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
CUDA_VISIBLE_DEVICES=0,1 DS_SKIP_CUDA_CHECK=1 accelerate launch \
  --config_file config/accelerate_kaggle_t4x2.yaml \
  --main_process_port 2222 \
  forget.py --config-name forget_po_forget5 \
  2>&1 | tee /kaggle/working/po_forget5_console.log
echo 'TRAIN_PROCESS_EXIT_0'

In [ ]:
from pathlib import Path
import torch
out = Path('/kaggle/working/models/llava_phi_PO_forget5')
checkpoint = out/'checkpoint.pt'
assert checkpoint.is_file(), '최종 checkpoint 없음. 콘솔 로그를 확인하세요.'
state = torch.load(checkpoint, map_location='cpu', weights_only=True)
assert state and all(torch.isfinite(t).all().item() for t in state.values())
print('최종 checkpoint:', checkpoint, 'MiB:', checkpoint.stat().st_size/1024**2)
print('저장 텐서:', len(state))
print('중간 checkpoint:', sorted(str(p) for p in out.glob('step_*/checkpoint.pt')))
del state
print('파일 존재 및 유한값 확인 완료. 논문 성능 재현 여부는 별도 evaluation으로 확인해야 합니다.')

학습 종료 후 모델 폴더, fiu_environment.txt, fiu_runtime.patch, fiu_data_revision.txt, po_forget5_console.log를 다운로드/보존하세요. GPU 메모리 통과와 논문 성능 재현은 별도입니다. 기존 scheduler 경고와 동결 구간 gradient 경고는 이 노트북에서 알고리즘을 변경해 숨기지 않습니다.